In [107]:
import pandas as pd

In [108]:
print(1)

1


In [109]:
# 1. Cargar datos (forma estándar, usa UTF-8 por defecto)
datos = pd.read_csv("SP1.csv")

# 2. Contar resultados absolutos (H/D/A)
conteo = datos['FTR'].value_counts()

# 3. Calcular porcentajes
porcentaje = datos['FTR'].value_counts(normalize=True) * 100


print("--- Porcentaje (%) ---")
print(porcentaje.round(2))

# 3. Convertir la cuota de local (B365H) a probabilidad
# Creamos una nueva columna 'Prob_Impl_H'
datos['Prob_Impl_A'] = (1 / datos['B365A']) * 100

# 4. Calcular la media de esa nueva columna
media_prob_implicitaA = datos['Prob_Impl_A'].mean()

# # Creamos una nueva columna 'Prob_Impl_H'
# datos['Prob_Impl_D'] = (1 / datos['B365D']) * 100

# # 4. Calcular la media de esa nueva columna
# media_prob_implicitaD = datos['Prob_Impl_D'].mean()

datos['beneficio_bet'] = (1 / datos['B365H']) * 100 + (1 / datos['B365D']) * 100 + (1 / datos['B365A']) * 100
media_beneficio = datos['beneficio_bet'].mean()

print("--- Probabilidad Implícita MEDIA (según B365) ---")
print(f"{media_prob_implicitaA.round(2)} %")

print("--- beneficio bet ---")
print(f"{media_beneficio.round(2)} %")
print(datos['beneficio_bet'])

--- Porcentaje (%) ---
FTR
H    45.56
D    27.78
A    26.67
Name: proportion, dtype: float64
--- Probabilidad Implícita MEDIA (según B365) ---
32.2 %
--- beneficio bet ---
105.52 %
0     105.516706
1     104.981203
2     105.714286
3     106.160751
4     105.202362
         ...    
85    105.054945
86    105.602241
87    105.491453
88    106.017223
89    105.555556
Name: beneficio_bet, Length: 90, dtype: float64


In [110]:

datos['beneficio_min_casas'] = (1 / datos['MaxH']) * 100 + (1 / datos['MaxD']) * 100 + (1 / datos['MaxA']) * 100


print("--- beneficio minimo casas de apuestas ---")

print(datos['beneficio_min_casas'])

for i in datos['beneficio_min_casas']:
    if i < 100:
        print(i)

--- beneficio minimo casas de apuestas ---
0     103.359037
1     102.801120
2     101.853282
3     103.018298
4     103.044872
         ...    
85    101.461869
86    102.374728
87    102.595525
88    102.480079
89    103.103172
Name: beneficio_min_casas, Length: 90, dtype: float64
99.49692953544044


ahora veamos el mercado +2.5 goles

In [111]:
# --- PASO 1: Calcular la Realidad del "Over 2.5" ---

# 1.1: Sumamos los goles para saber el total de cada partido
datos['Goles_Totales'] = datos['FTHG'] + datos['FTAG']

# 1.2: Creamos una columna que es 'True' si hay más de 2.5 goles
datos['Es_Over_2_5'] = datos['Goles_Totales'] > 2.5

# 1.3: Calculamos el porcentaje real de partidos con Over 2.5
# (En Python, True=1 y False=0. Hacer la media nos da el porcentaje)
porcentaje_real_over = datos['Es_Over_2_5'].mean() * 100

print(f"--- Porcentaje REAL de partidos 'Over 2.5' ---")
print(f"{porcentaje_real_over.round(2)} %")
print("\n")

# --- PASO 2: Calcular la Probabilidad Implícita ---

# 2.1: Convertimos la cuota 'B365>2.5' (el Over) a probabilidad
# (El nombre de la columna en el CSV es 'B365>2.5')
datos['Prob_Impl_Over'] = (1 / datos['B365>2.5']) * 100

# 2.2: Calculamos la media de esa probabilidad implícita
media_prob_impl_over = datos['Prob_Impl_Over'].mean()

print(f"--- Probabilidad Implícita MEDIA de 'Over 2.5' (B365) ---")
print(f"{media_prob_impl_over.round(2)} %")

--- Porcentaje REAL de partidos 'Over 2.5' ---
47.78 %


--- Probabilidad Implícita MEDIA de 'Over 2.5' (B365) ---
50.46 %


In [112]:
porcentaje_real_menos = 100 - porcentaje_real_over # porque el porcentaje ficticio si debe sumar 100%

print(f"--- Porcentaje REAL de partidos 'menos 2.5' ---")
print(f"{porcentaje_real_menos.round(2)} %")
print("\n")

# --- PASO 2: Calcular la Probabilidad Implícita ---

# 2.1: Convertimos la cuota 'B365>2.5' (el Over) a probabilidad
# (El nombre de la columna en el CSV es 'B365>2.5')
datos['Prob_Impl_menos'] = (1 / datos['B365<2.5']) * 100

# 2.2: Calculamos la media de esa probabilidad implícita
media_prob_impl_menos = datos['Prob_Impl_menos'].mean()

print(f"--- Probabilidad Implícita MEDIA de 'Over 2.5' (B365) ---")
print(f"{media_prob_impl_menos.round(2)} %")

--- Porcentaje REAL de partidos 'menos 2.5' ---
52.22 %


--- Probabilidad Implícita MEDIA de 'Over 2.5' (B365) ---
54.54 %


ordenar por fecha todos los partidos 

In [113]:
datos['Date'] = pd.to_datetime(datos['Date'], dayfirst=True)
datos = datos.sort_values(by='Date')

separamos entre local y visitante


In [114]:

# 3. Convertir FTR a Puntos
mapa_puntos_local = {'H': 3, 'D': 1, 'A': 0}
mapa_puntos_visitante = {'H': 0, 'D': 1, 'A': 3}
datos['Puntos_Local'] = datos['FTR'].map(mapa_puntos_local)
datos['Puntos_Visitante'] = datos['FTR'].map(mapa_puntos_visitante)

# --- NUEVO PASO: CALCULAR FORMA y DIFICULTAD (Tu idea) ---

# 4. Calcular probabilidades PROPIAS de ganar (H y A)
datos['Prob_H'] = (1 / datos['B365H']) * 100
datos['Prob_A'] = (1 / datos['B365A']) * 100

# 5. Crear tabla "larga" de equipos (AHORA CON PROBABILIDAD PROPIA)
datos_local = datos[['Date', 'HomeTeam', 'Puntos_Local', 'Prob_H']].rename(
    columns={'HomeTeam': 'Equipo', 'Puntos_Local': 'Puntos', 'Prob_H': 'Prob_Propia'}
)
datos_visitante = datos[['Date', 'AwayTeam', 'Puntos_Visitante', 'Prob_A']].rename(
    columns={'AwayTeam': 'Equipo', 'Puntos_Visitante': 'Puntos', 'Prob_A': 'Prob_Propia'}
)

# Los juntamos en una sola tabla
datos_equipo = pd.concat([datos_local, datos_visitante])
datos_equipo = datos_equipo.sort_values(by=['Equipo', 'Date'])


# 6. Calcular la forma (media de puntos) Y la probabilidad propia
datos_equipo['Forma_Puntos'] = datos_equipo.groupby('Equipo')['Puntos'].shift(1).rolling(window=5, min_periods=1).mean()
# ESTA ES LA LÍNEA NUEVA PARA TU IDEA:
datos_equipo['Forma_Prob_Propia'] = datos_equipo.groupby('Equipo')['Prob_Propia'].shift(1).rolling(window=5, min_periods=1).mean()


# 7. Unir (Merge) ambas columnas de forma de vuelta a la tabla original 'datos'
columnas_a_unir = ['Date', 'Equipo', 'Forma_Puntos', 'Forma_Prob_Propia']

# Primero para el Local
datos = pd.merge(
    datos,
    datos_equipo[columnas_a_unir],
    left_on=['Date', 'HomeTeam'],
    right_on=['Date', 'Equipo'],
    how='left'
)
datos = datos.rename(columns={
    'Forma_Puntos': 'Forma_Puntos_Local',
    'Forma_Prob_Propia': 'Forma_Prob_Propia_Local'
})

# Luego para el Visitante
datos = pd.merge(
    datos,
    datos_equipo[columnas_a_unir],
    left_on=['Date', 'AwayTeam'],
    right_on=['Date', 'Equipo'],
    how='left'
)
datos = datos.rename(columns={
    'Forma_Puntos': 'Forma_Puntos_Visitante',
    'Forma_Prob_Propia': 'Forma_Prob_Propia_Visitante'
})


# 8. Imprimir el resultado para verificar
columnas_a_mostrar = [
    'Date',
    'HomeTeam',
    'Forma_Puntos_Local',
    'Forma_Prob_Propia_Local', # Nueva columna
    'AwayTeam',
    'Forma_Puntos_Visitante',
    'Forma_Prob_Propia_Visitante' # Nueva columna
]
print("--- Datos con 'Forma' y 'Probabilidad Propia' (Tu idea) ---")
# Usamos .tail() para ver los datos de final de temporada, que ya tendrán valores
print(datos[columnas_a_mostrar].tail(10).round(2)) # Añadido .round(2) para limpiar

--- Datos con 'Forma' y 'Probabilidad Propia' (Tu idea) ---
         Date    HomeTeam  Forma_Puntos_Local  Forma_Prob_Propia_Local     AwayTeam  Forma_Puntos_Visitante  Forma_Prob_Propia_Visitante
80 2025-10-17      Oviedo                 0.6                    22.53      Espanol                     1.0                        34.50
81 2025-10-18  Ath Madrid                 2.2                    53.97      Osasuna                     1.4                        35.48
82 2025-10-18     Sevilla                 2.0                    31.45     Mallorca                     0.8                        25.29
83 2025-10-18   Barcelona                 2.4                    78.33       Girona                     1.2                        33.87
84 2025-10-18  Villarreal                 1.8                    39.07        Betis                     2.0                        44.87
85 2025-10-19     Levante                 1.6                    25.64    Vallecano                     0.8           

In [115]:
import pandas as pd

# Opciones de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 1. Cargar datos
datos = pd.read_csv("SP1.csv")

# --- FASE 1: PREPARACIÓN ---

# 2. Ordenar por fecha
datos['Date'] = pd.to_datetime(datos['Date'], dayfirst=True)
datos = datos.sort_values(by='Date')

# 3. Convertir FTR a Puntos
mapa_puntos_local = {'H': 3, 'D': 1, 'A': 0}
mapa_puntos_visitante = {'H': 0, 'D': 1, 'A': 3}
datos['Puntos_Local'] = datos['FTR'].map(mapa_puntos_local)
datos['Puntos_Visitante'] = datos['FTR'].map(mapa_puntos_visitante)

# 4. Calcular probabilidades PROPIAS (Tu idea de Dificultad)
datos['Prob_H'] = ((1 / datos['B365H']) * 100).clip(lower=1)
datos['Prob_A'] = ((1 / datos['B365A']) * 100).clip(lower=1)


# --- FASE 2: CREACIÓN DE VARIABLES (Feature Engineering) ---

# 5. Crear tabla "larga" de equipos (con Puntos, Prob, Goles y Tiros)
datos_local = datos[['Date', 'HomeTeam', 'Puntos_Local', 'Prob_H', 'FTHG', 'FTAG', 'HST', 'AST']].rename(
    columns={
        'HomeTeam': 'Equipo', 'Puntos_Local': 'Puntos', 'Prob_H': 'Prob_Propia',
        'FTHG': 'Goles_Favor', 'FTAG': 'Goles_Contra',
        'HST': 'Tiros_Puerta_Favor', 'AST': 'Tiros_Puerta_Contra'
    }
)
datos_visitante = datos[['Date', 'AwayTeam', 'Puntos_Visitante', 'Prob_A', 'FTAG', 'FTHG', 'AST', 'HST']].rename(
    columns={
        'AwayTeam': 'Equipo', 'Puntos_Visitante': 'Puntos', 'Prob_A': 'Prob_Propia',
        'FTAG': 'Goles_Favor', 'FTHG': 'Goles_Contra',
        'AST': 'Tiros_Puerta_Favor', 'HST': 'Tiros_Puerta_Contra'
    }
)

datos_equipo = pd.concat([datos_local, datos_visitante])
datos_equipo = datos_equipo.sort_values(by=['Equipo', 'Date'])

# 6. Calcular TODAS las métricas de forma (rolling 5 partidos)
columnas_a_promediar = [
    'Puntos', 'Prob_Propia', 'Goles_Favor', 'Goles_Contra',
    'Tiros_Puerta_Favor', 'Tiros_Puerta_Contra'
]
forma_equipo = datos_equipo.groupby('Equipo')[columnas_a_promediar].shift(1).rolling(window=5, min_periods=1).mean()

# Renombramos las columnas
forma_equipo = forma_equipo.rename(columns={
    'Puntos': 'Forma_Puntos', 'Prob_Propia': 'Forma_Prob_Propia',
    'Goles_Favor': 'Forma_Goles_Favor', 'Goles_Contra': 'Forma_Goles_Contra',
    'Tiros_Puerta_Favor': 'Forma_Tiros_Puerta_Favor',
    'Tiros_Puerta_Contra': 'Forma_Tiros_Puerta_Contra'
})

# 7. Unir (Merge) las columnas de forma de vuelta a 'datos_equipo'
datos_equipo = pd.concat([datos_equipo.reset_index(drop=True), forma_equipo.reset_index(drop=True)], axis=1)

# 8. Unir (Merge) la forma de vuelta a la tabla original 'datos'
columnas_a_unir = [
    'Date', 'Equipo',
    'Forma_Puntos', 'Forma_Prob_Propia', 'Forma_Goles_Favor', 'Forma_Goles_Contra',
    'Forma_Tiros_Puerta_Favor', 'Forma_Tiros_Puerta_Contra'
]
datos = pd.merge(
    datos, datos_equipo[columnas_a_unir],
    left_on=['Date', 'HomeTeam'], right_on=['Date', 'Equipo'],
    how='left', suffixes=('_OLD', '_Local')
)
datos = pd.merge(
    datos, datos_equipo[columnas_a_unir],
    left_on=['Date', 'AwayTeam'], right_on=['Date', 'Equipo'],
    how='left', suffixes=('_Local_Final', '_Visitante')
)

# 9. Imprimir el resultado (verificación final de Fase 2)
columnas_a_mostrar = [
    'HomeTeam',
    'Forma_Tiros_Puerta_Favor_Local_Final',
    'Forma_Prob_Propia_Local_Final',
    'AwayTeam',
    'Forma_Tiros_Puerta_Favor_Visitante',
    'Forma_Prob_Propia_Visitante'
]

print("--- FASE 2 COMPLETADA ---")
print(datos[columnas_a_mostrar].tail(5).round(2))

--- FASE 2 COMPLETADA ---
   HomeTeam  Forma_Tiros_Puerta_Favor_Local_Final  Forma_Prob_Propia_Local_Final     AwayTeam  Forma_Tiros_Puerta_Favor_Visitante  Forma_Prob_Propia_Visitante
85  Levante                                   3.8                          25.64    Vallecano                                 4.4                        34.64
86   Getafe                                   2.4                          35.25  Real Madrid                                 6.0                        66.69
87    Elche                                   4.4                          32.54   Ath Bilbao                                 4.4                        52.61
88    Celta                                   4.0                          37.22     Sociedad                                 3.6                        32.20
89   Alaves                                   3.8                          32.50     Valencia                                 3.8                        33.27


ahora comenzemos con modelos de machine learning


ideas: usar goles esperados de un partido para dar un resultado ficticio pero muchas veces mas cercano al resultado q deberia haber sido referentemente al nivel del partido

errores q voy teniendo: probe a multiplicar el numero de tiros a puerta * cuota pero favorecia demasiado a equips bajos q un simple tira y subia en exceso la media
